# ⚡ Priority Queues — Sorting con PQ y Colas de Prioridad Adaptables

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap09/03_Sorting_Heaps_Adaptables_Teoria.ipynb)


### 🎯 Objetivos

1. ✅ Implementar **Selection-Sort** e **Insertion-Sort** como aplicaciones de la PQ
2. ✅ Implementar **Heap-Sort** in-place en O(n log n)
3. ✅ Comprender el concepto de **Locator** para colas adaptables
4. ✅ Implementar `AdaptableHeapPriorityQueue` con `update()` y `remove()` en O(log n)
5. ✅ Aplicar colas adaptables en algoritmos de grafos (contexto)

📚 *Data Structures and Algorithms in Python* — Goodrich, Tamassia, Goldwasser (Cap. 9.4–9.5)  


---

## 📌 9.4 — Sorting con Priority Queues

### El algoritmo `pq_sort`

```python
def pq_sort(C, P):
    """Sort a list C using priority queue P.
    C: colección con elementos
    P: priority queue vacía
    """
    n = len(C)
    for j in range(n):
        element = C.pop()
        P.add(element, element)     # Insertar en PQ (key = value)
    for j in range(n):
        (k, v) = P.remove_min()     # Extraer en orden
        C.append(v)
```

La variante de PQ determina el algoritmo de sorting:

| PQ Usada | Algoritmo | Complejidad |
|----------|-----------|-------------|
| `UnsortedPriorityQueue` | Selection-Sort | O(n²) |
| `SortedPriorityQueue` | Insertion-Sort | O(n²) worst / O(n) best |
| `HeapPriorityQueue` | Heap-Sort | **O(n log n)** |


In [ ]:
# Re-importar clases base necesarias
class Empty(Exception):
    pass

class PriorityQueueBase:
    class _Item:
        __slots__ = '_key', '_value'
        def __init__(self, k, v): self._key = k; self._value = v
        def __lt__(self, other): return self._key < other._key
        def __repr__(self): return f'({self._key},{self._value!r})'
    def is_empty(self): return len(self) == 0

class UnsortedPriorityQueue(PriorityQueueBase):
    def __init__(self): self._data = []
    def __len__(self): return len(self._data)
    def _find_min(self):
        if self.is_empty(): raise Empty()
        return min(range(len(self._data)), key=lambda i: self._data[i])
    def add(self, k, v): self._data.append(self._Item(k, v))
    def min(self):
        i = self._find_min(); item = self._data[i]; return (item._key, item._value)
    def remove_min(self):
        i = self._find_min(); item = self._data.pop(i); return (item._key, item._value)

class SortedPriorityQueue(PriorityQueueBase):
    def __init__(self): self._data = []
    def __len__(self): return len(self._data)
    def add(self, k, v):
        item = self._Item(k, v)
        i = len(self._data)
        while i > 0 and item < self._data[i-1]: i -= 1
        self._data.insert(i, item)
    def min(self):
        if self.is_empty(): raise Empty()
        item = self._data[0]; return (item._key, item._value)
    def remove_min(self):
        if self.is_empty(): raise Empty()
        item = self._data.pop(0); return (item._key, item._value)

class HeapPriorityQueue(PriorityQueueBase):
    def _parent(self, j): return (j-1)//2
    def _left(self, j): return 2*j+1
    def _right(self, j): return 2*j+2
    def _has_left(self, j): return self._left(j) < len(self._data)
    def _has_right(self, j): return self._right(j) < len(self._data)
    def _swap(self, i, j): self._data[i], self._data[j] = self._data[j], self._data[i]
    def _upheap(self, j):
        p = self._parent(j)
        if j > 0 and self._data[j] < self._data[p]:
            self._swap(j, p); self._upheap(p)
    def _downheap(self, j):
        if self._has_left(j):
            l = self._left(j); s = l
            if self._has_right(j):
                r = self._right(j)
                if self._data[r] < self._data[l]: s = r
            if self._data[s] < self._data[j]:
                self._swap(j, s); self._downheap(s)
    def __init__(self, contents=()):
        self._data = [self._Item(k,v) for k,v in contents]
        if len(self._data) > 1:
            for j in range(self._parent(len(self._data)-1), -1, -1):
                self._downheap(j)
    def __len__(self): return len(self._data)
    def min(self):
        if self.is_empty(): raise Empty()
        item = self._data[0]; return (item._key, item._value)
    def add(self, k, v):
        self._data.append(self._Item(k,v)); self._upheap(len(self._data)-1)
    def remove_min(self):
        if self.is_empty(): raise Empty()
        self._swap(0, len(self._data)-1); item = self._data.pop()
        self._downheap(0); return (item._key, item._value)

print('✅ Clases PQ cargadas')


---

## 📌 9.4.1 — Selection-Sort (con UnsortedPriorityQueue)

### Análisis

**Fase 1 (insertar n elementos):** Cada `add` es O(1) → total **O(n)**  
**Fase 2 (extraer n elementos):** Cada `remove_min` es O(n) → total **O(n²)**

- **Peor caso:** O(n²)
- **Mejor caso:** O(n²) — siempre debe buscar el mínimo
- **Espacio:** O(n)

```
Para [7, 4, 8, 2, 5]:
Fase 2: encuentra 2 (busca todo) → 4 → 5 → 7 → 8
```


In [ ]:
def selection_sort(data):
    """Sort list using Selection-Sort via UnsortedPriorityQueue. O(n²)."""
    P = UnsortedPriorityQueue()
    for x in data:
        P.add(x, x)         # O(1) por inserción
    return [P.remove_min()[1] for _ in range(len(data))]   # O(n) por extracción


# Demostración con traza manual
print('=== Selection-Sort via UnsortedPriorityQueue ===')
data = [7, 4, 8, 2, 5, 1, 9, 3, 6]
print(f'Input:  {data}')
result = selection_sort(data[:])
print(f'Output: {result}')
print(f'¿Ordenado? {result == sorted(data)}')
print()

# Traza de la búsqueda del mínimo
P_trace = UnsortedPriorityQueue()
for x in [7, 4, 8, 2, 5]:
    P_trace.add(x, x)
print('Traza extracción con lista no ordenada:')
step = 1
while not P_trace.is_empty():
    print(f'  Paso {step}: cola={P_trace._data}, remove_min()={P_trace.remove_min()}')
    step += 1


---

## 📌 9.4.2 — Insertion-Sort (con SortedPriorityQueue)

### Análisis

**Fase 1 (insertar n elementos):** Cada `add` puede ser O(n) → total **O(n²)**  
**Fase 2 (extraer n elementos):** Cada `remove_min` es O(1) → total **O(n)**

- **Peor caso:** O(n²) — lista en orden inverso
- **Mejor caso:** O(n) — lista ya ordenada (cada insert es O(1))
- **Espacio:** O(n)

> **Insight:** Insertion-Sort es eficiente con datos casi ordenados.


In [ ]:
def insertion_sort(data):
    """Sort list using Insertion-Sort via SortedPriorityQueue. O(n²) worst."""
    P = SortedPriorityQueue()
    for x in data:
        P.add(x, x)         # O(n) por inserción (mantiene orden)
    return [P.remove_min()[1] for _ in range(len(data))]   # O(1) por extracción


print('=== Insertion-Sort via SortedPriorityQueue ===')
data = [7, 4, 8, 2, 5, 1, 9, 3, 6]
print(f'Input:  {data}')
result = insertion_sort(data[:])
print(f'Output: {result}')
print(f'¿Ordenado? {result == sorted(data)}')
print()

# Mejor caso: lista casi ordenada
import random
import time

# Comparar tiempos: random vs casi-ordenado
N = 1000
random_data = list(range(N))
random.shuffle(random_data)
almost_sorted = list(range(N))
# Perturbar solo 10 elementos
for _ in range(10):
    i, j = random.randint(0, N-1), random.randint(0, N-1)
    almost_sorted[i], almost_sorted[j] = almost_sorted[j], almost_sorted[i]

t0 = time.perf_counter(); insertion_sort(random_data[:])
t_rand = time.perf_counter() - t0

t0 = time.perf_counter(); insertion_sort(almost_sorted[:])
t_sorted = time.perf_counter() - t0

print(f'Insertion-Sort con N={N}:')
print(f'  Datos aleatorios: {t_rand*1000:.2f} ms')
print(f'  Casi ordenados:   {t_sorted*1000:.2f} ms')
print(f'  Speedup: {t_rand/t_sorted:.1f}x (ventaja del mejor caso)')


---

## 📌 9.4.3 — Heap-Sort

### Dos fases con HeapPriorityQueue

**Fase 1:** Insertar los n elementos → n × O(log n) = **O(n log n)**  
**Fase 2:** Extraer n veces con remove_min → n × O(log n) = **O(n log n)**  
**Total:** **O(n log n)**

### Heap-Sort In-Place

Podemos hacer heap-sort **sin espacio adicional** usando el mismo array:

1. **Fase 1 (construcción):** Transformar el array en un **max-heap** in-place usando bottom-up
2. **Fase 2 (ordenamiento):** Repetidamente extraer el máximo y colocarlo al final

> **Proposición 9.4:** Heap-Sort realiza O(n log n) comparaciones.


In [ ]:
def heap_sort_inplace(data):
    """In-place heap sort. O(n log n) time, O(1) extra space."""
    n = len(data)
    
    # Fase 1: Construir max-heap in-place (bottom-up)
    # Para max-heap, _downheap pide que padre >= hijos
    def max_downheap(arr, j, size):
        """Downheap for max-heap on arr[0..size-1] starting at j."""
        largest = j
        l = 2 * j + 1
        r = 2 * j + 2
        if l < size and arr[l] > arr[largest]:
            largest = l
        if r < size and arr[r] > arr[largest]:
            largest = r
        if largest != j:
            arr[j], arr[largest] = arr[largest], arr[j]
            max_downheap(arr, largest, size)
    
    # Construir max-heap
    for j in range((n - 1) // 2, -1, -1):
        max_downheap(data, j, n)
    
    # Fase 2: Extraer max repetidamente al final del array
    for i in range(n - 1, 0, -1):
        data[0], data[i] = data[i], data[0]   # máximo al final
        max_downheap(data, 0, i)               # restaurar heap en [0..i-1]
    
    return data


# Demostración
print('=== Heap-Sort In-Place ===')
data = [7, 4, 8, 2, 5, 1, 9, 3, 6]
print(f'Input:  {data}')
result = heap_sort_inplace(data[:])
print(f'Output: {result}')
print(f'¿Ordenado? {result == sorted(data)}')

# Comparación de rendimiento
import time, random

def benchmark_sort(sort_func, n, seed=42):
    random.seed(seed)
    data = [random.randint(1, n) for _ in range(n)]
    t0 = time.perf_counter()
    sort_func(data[:])
    return (time.perf_counter() - t0) * 1000

print()
print(f'Comparación de sorting (ms) para N=5000:')
N = 5000
print(f'  heap_sort_inplace: {benchmark_sort(heap_sort_inplace, N):.2f} ms')
print(f'  selection_sort:    {benchmark_sort(selection_sort, N):.2f} ms')
print(f'  Python sorted():   ', end='')
import random; random.seed(42); d = [random.randint(1,N) for _ in range(N)]
t0 = time.perf_counter(); sorted(d); print(f'{(time.perf_counter()-t0)*1000:.2f} ms')


---

## 📊 Comparación Final de Algoritmos de Sorting via PQ

### Tabla 9.3 del libro

| Algoritmo | PQ Usada | Mejor caso | Peor caso | In-Place |
|-----------|----------|:----------:|:---------:|:--------:|
| Selection-Sort | UnsortedPQ | **O(n²)** | **O(n²)** | ✗ |
| Insertion-Sort | SortedPQ | **O(n)** | **O(n²)** | ✗ |
| Heap-Sort | HeapPQ | **O(n log n)** | **O(n log n)** | ✅ |

> Heap-Sort domina en el peor caso sin sacrificar espacio.


---

## 📌 9.5 — Colas de Prioridad Adaptables

### Motivación

En algunos algoritmos (ej: Dijkstra, Prim) necesitamos:
- **`remove(loc)`** → Eliminar un elemento arbitrario (no solo el mínimo)
- **`update(loc, k, v)`** → Cambiar la clave de un elemento existente

Esto requiere un mecanismo para **localizar** elementos arbitrarios eficientemente.

### Locators (Posicionadores)

Un **Locator** es un token que:
1. Se obtiene al insertar: `loc = P.add(k, v)` retorna un Locator
2. El usuario lo guarda como referencia
3. Permite operaciones directas: `P.update(loc, new_k, new_v)`, `P.remove(loc)`

```python
loc_a = P.add(5, 'A')   # → Locator para (5,'A')
loc_b = P.add(3, 'B')   # → Locator para (3,'B')
P.update(loc_a, 1, 'A') # Cambia clave de 5 → 1
P.remove(loc_b)          # Elimina (3,'B') directamente
```

### Complejidad

| Operación | Complejidad |
|-----------|-------------|
| `add(k, v)` | O(log n) |
| `min()` | O(1) |
| `remove_min()` | O(log n) |
| **`update(loc, k, v)`** | **O(log n)** |
| **`remove(loc)`** | **O(log n)** |


In [ ]:
class AdaptableHeapPriorityQueue(HeapPriorityQueue):
    """A locator-based priority queue implemented with a binary heap."""

    # ─── Nested Locator class ───────────────────────────────────────────────
    class Locator(HeapPriorityQueue._Item):
        """Token for locating an entry of the priority queue."""
        __slots__ = '_index'     # Índice actual en el array heap

        def __init__(self, k, v, j):
            super().__init__(k, v)
            self._index = j

    # ─── Nonpublic behaviors ────────────────────────────────────────────────
    def _swap(self, i, j):
        """Override swap to also update locator indices."""
        super()._swap(i, j)
        # Actualizar índices después del swap
        self._data[i]._index = i
        self._data[j]._index = j

    def _bubble(self, j):
        """Move item at index j to restore heap-order (up or down)."""
        if j > 0 and self._data[j] < self._data[self._parent(j)]:
            self._upheap(j)
        else:
            self._downheap(j)

    # ─── Public behaviors ───────────────────────────────────────────────────
    def add(self, key, value):
        """Add a key-value pair. Return locator token. O(log n)."""
        token = self.Locator(key, value, len(self._data))
        self._data.append(token)
        self._upheap(len(self._data) - 1)
        return token     # ← retorna el Locator al usuario

    def update(self, loc, newkey, newval):
        """Update the key and value for the entry identified by Locator. O(log n)."""
        j = loc._index
        if not (0 <= j < len(self) and self._data[j] is loc):
            raise ValueError('Invalid locator')
        loc._key = newkey
        loc._value = newval
        self._bubble(j)

    def remove(self, loc):
        """Remove and return the (k,v) pair identified by Locator. O(log n)."""
        j = loc._index
        if not (0 <= j < len(self) and self._data[j] is loc):
            raise ValueError('Invalid locator')
        if j == len(self) - 1:     # Si es el último, simplemente sacarlo
            self._data.pop()
        else:
            self._swap(j, len(self) - 1)   # Mover al final
            self._data.pop()               # Sacar del final
            self._bubble(j)                # Restaurar heap-order
        return (loc._key, loc._value)


# ── Demostración ──
print('=== AdaptableHeapPriorityQueue ===')
P = AdaptableHeapPriorityQueue()

loc_a = P.add(5, 'A')
loc_b = P.add(3, 'B')
loc_c = P.add(8, 'C')
loc_d = P.add(1, 'D')
loc_e = P.add(7, 'E')

print(f'Estado inicial: {P._data}')
print(f'min() = {P.min()}')
print()

# Actualizar prioridad de A: 5 → 0 (ahora es el más urgente)
P.update(loc_a, 0, 'A-URGENTE')
print(f'Después de update(loc_a, 0, A-URGENTE): {P._data}')
print(f'min() = {P.min()}')
print()

# Eliminar C directamente
removed = P.remove(loc_c)
print(f'remove(loc_c) → {removed}')
print(f'Estado: {P._data}')
print()

# Extraer en orden
print('Extrayendo en orden:')
while not P.is_empty():
    print(f'  remove_min() → {P.remove_min()}')


---

## 🗺️ Aplicación: Contexto de Dijkstra

La `AdaptableHeapPriorityQueue` es esencial para la implementación eficiente de:

### Algoritmo de Dijkstra (shortest paths)

```python
# Pseudocódigo con cola adaptable
def dijkstra(graph, src):
    dist = {v: float('inf') for v in graph}
    dist[src] = 0
    
    pq = AdaptableHeapPriorityQueue()
    locators = {}
    
    for v in graph:
        locators[v] = pq.add(dist[v], v)   # Guarda locator
    
    while not pq.is_empty():
        d, u = pq.remove_min()
        for neighbor, weight in graph[u]:
            new_dist = d + weight
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                pq.update(locators[neighbor], new_dist, neighbor)  # O(log n)!
    
    return dist
```

Sin `update`, tendríamos que eliminar e insertar → O(n) en vez de O(log n)  
Con `AdaptableHeapPriorityQueue`: Dijkstra corre en **O((V + E) log V)**


---

## 📝 Resumen del Notebook 3

### Sorting con Priority Queues

| Algoritmo | PQ | add | remove_min | Total | In-place |
|-----------|-----|-----|-----------|-------|---------|
| **Selection-Sort** | Unsorted | O(1) | O(n) | **O(n²)** | ✗ |
| **Insertion-Sort** | Sorted | O(n) | O(1) | **O(n²)** worst | ✗ |
| **Heap-Sort** | Heap | O(log n) | O(log n) | **O(n log n)** | ✅ |

### AdaptableHeapPriorityQueue

| Operación | Complejidad | Uso |
|-----------|-------------|-----|
| `add(k,v)` → Locator | O(log n) | Dijkstra: inicializar |
| `update(loc, k, v)` | O(log n) | Dijkstra: relajar aristas |
| `remove(loc)` | O(log n) | Prim: actualizar árbol |

### Jerarquía de Clases del Capítulo 9

```
PriorityQueueBase
├── UnsortedPriorityQueue    (add O(1), remove_min O(n))
├── SortedPriorityQueue      (add O(n), remove_min O(1))
└── HeapPriorityQueue        (add O(logn), remove_min O(logn))
    └── AdaptableHeapPriorityQueue  (+ update/remove con Locator)
```

---

🔗 **Referencia:** Goodrich et al., Cap. 9.4–9.5

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>